# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print general dataset metadata summary
metadata = dataset.metadata # MetadataObject
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.date_published}")
print(f"Keywords: {', '.join(metadata.keywords)}")
print(f"License: {metadata.license}")
print(f"Cite as: {metadata.cite_as}")


## 2. Data Overview
Review available record sets, fields, and their IDs. Each entity is referenced by its `@id` for consistency.

In [ ]:
print("### Record Sets:")
for rs in dataset.metadata.record_sets:
    print(f"ID: {rs.id} | Name: {rs.name} | Description: {rs.description}")
    print("  Fields and Columns:")
    for f in rs.fields:
        # Columns live within fields and may contain source-specific mapping
        col_ids = [col.id for col in f.columns] if hasattr(f, 'columns') else []
        print(f"    Field ID: {f.id} | Name: {f.name} | Data type: {f.data_type}")
        for cid in col_ids:
            print(f"      Column ID: {cid}")
    print()


## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis. Use the record set and field `@id`s from the overview.

> For this example, we'll extract the main protein abundance table. Typically, this will be the primary data table - please consult above printout for other record sets.

In [ ]:
# Identify all RecordSet @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Available RecordSet IDs: {record_set_ids}")

# We'll use the first record set as the primary data table for demonstration
main_record_set_id = record_set_ids[0]

dataframes = {}
for rsid in record_set_ids:
    print(f"Loading records from RecordSet: {rsid}")
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for {rsid}\n")

print('Column names for the main data table:')
print(dataframes[main_record_set_id].columns.to_list())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Use field `@id` as references and keep operations dynamic. This section demonstrates outlier removal, normalization, and grouping.

In [ ]:
# Inspect numeric fields (by their @id as column)
df = dataframes[main_record_set_id]
print('First few column names:', df.columns.to_list())

# Select a numeric field for analysis (use field @id from Data Overview section printed above)
# You may have fields like 'cr:peptide_count' or 'cr:MW' - replace below if needed
numeric_field = None
for col in df.columns:
    if 'MW' in col or 'peptide_count' in col or 'abundance' in col or 'coverage' in col:
        numeric_field = col
        break

if numeric_field is None:
    print('No numeric field found - please check column names and dataset schema.')
else:
    # Proceed with the field
    print(f"Using numeric field: {numeric_field}")
    # Convert column to numeric, coercing errors
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as demo threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (75th percentile): {len(filtered_df)} records")
    
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by a categorical field (e.g. accession or protein_id or sample)
    group_field = None
    for col in df.columns:
        if 'accession' in col or 'protein' in col or 'sample' in col:
            group_field = col
            break

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} grouped by {group_field} (top 5):")
        print(grouped_df.head())
    else:
        print('No suitable group field found for grouping.')

## 5. Visualization
Visualize the distribution of a numeric field and highlight high-abundance examples.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # Plot the normalized field for filtered records if previously computed
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=20, kde=True, color='darkorange')
        plt.title(f"Normalized {numeric_field} distribution (filtered, >75th percentile)")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.show()


## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" via the `mlcroissant` library. We demonstrated how to inspect metadata, explore record sets and fields by their `@id`, extract tabular data, filter and normalize values, and quickly visualize key distributions for downstream analysis. This workflow can be adapted for any Croissant-compliant dataset for FAIR and reproducible data science.
